# 📈 Forecasting de Ventas 2025

Este notebook importa las librerías principales y carga el archivo de inferencia para realizar predicciones de ventas.

In [2]:
# 🤖 Importación de librerías principales con alias estándar
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import model_selection, metrics, preprocessing
import holidays

## Carga de datos de inferencia
Cargamos el archivo de ventas de 2025 para inferencia en un dataframe llamado `inferencia_df`.

In [3]:
# Carga del archivo de inferencia
data_path = '../data/raw/inferencia/ventas_2025_inferencia.csv'
inferencia_df = pd.read_csv(data_path)
print('Shape de inferencia_df:', inferencia_df.shape)
inferencia_df.head()

Shape de inferencia_df: (888, 13)


,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,Amazon,Decathlon,Deporvillage
0,2025-10-25,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,26.0,113.13,2941.38,89.51,113.43,104.78
1,2025-10-25,PROD_002,Adidas Ultraboost 23,Running,Zapatillas Running,135,True,27.0,141.89,3831.03,128.73,112.91,122.88
2,2025-10-25,PROD_003,Asics Gel Nimbus 25,Running,Zapatillas Running,85,False,5.0,85.79,428.95,84.28,74.51,85.57
3,2025-10-25,PROD_004,New Balance Fresh Foam X 1080v12,Running,Zapatillas Running,75,False,3.0,76.19,228.57,75.54,70.32,71.13
4,2025-10-25,PROD_005,Nike Dri-FIT Miler,Running,Ropa Running,35,False,3.0,35.48,106.44,33.84,31.32,34.41


## Preparación y transformación de inferencia_df
Aplicamos exactamente las mismas transformaciones, ingeniería de variables y codificaciones que en el notebook de entrenamiento para dejar inferencia_df listo para la inferencia con el modelo.

In [4]:
# Conversión de la columna 'fecha' a datetime
inferencia_df['fecha'] = pd.to_datetime(inferencia_df['fecha'])

# Variables temporales y de calendario
inferencia_df['año'] = inferencia_df['fecha'].dt.year
inferencia_df['mes'] = inferencia_df['fecha'].dt.month
inferencia_df['dia_mes'] = inferencia_df['fecha'].dt.day
inferencia_df['dia_semana'] = inferencia_df['fecha'].dt.dayofweek  # 0=Lunes
inferencia_df['es_fin_de_semana'] = inferencia_df['dia_semana'].isin([5,6])

# Festivos nacionales en España
es_holidays = holidays.country_holidays('ES', years=inferencia_df['año'].unique())
inferencia_df['es_festivo'] = inferencia_df['fecha'].isin(es_holidays)

# Black Friday: último viernes de noviembre
def es_black_friday(fecha):
    if fecha.month != 11:
        return False
    ultimo_viernes = max([d for d in pd.date_range(start=f'{fecha.year}-11-01', end=f'{fecha.year}-11-30') if d.weekday() == 4])
    return fecha == ultimo_viernes
inferencia_df['es_black_friday'] = inferencia_df['fecha'].apply(es_black_friday)

# Cyber Monday: primer lunes después de Black Friday
def es_cyber_monday(fecha):
    if fecha.month != 11 and fecha.month != 12:
        return False
    ultimo_viernes = max([d for d in pd.date_range(start=f'{fecha.year}-11-01', end=f'{fecha.year}-11-30') if d.weekday() == 4])
    cyber_monday = ultimo_viernes + pd.Timedelta(days=3)
    return fecha == cyber_monday
inferencia_df['es_cyber_monday'] = inferencia_df['fecha'].apply(es_cyber_monday)

# Variables adicionales
def _month_end(fecha):
    return fecha + pd.offsets.MonthEnd(0)
inferencia_df['semana_año'] = inferencia_df['fecha'].dt.isocalendar().week
inferencia_df['dia_año'] = inferencia_df['fecha'].dt.dayofyear
inferencia_df['es_primer_dia_mes'] = inferencia_df['dia_mes'] == 1
inferencia_df['es_ultimo_dia_mes'] = inferencia_df['fecha'] == inferencia_df['fecha'].apply(_month_end)
inferencia_df['trimestre'] = inferencia_df['fecha'].dt.quarter
inferencia_df['es_festivo_o_finde'] = inferencia_df['es_festivo'] | inferencia_df['es_fin_de_semana']

In [5]:
# Variable descuento porcentaje
inferencia_df['descuento_porcentaje'] = ((inferencia_df['precio_venta'] - inferencia_df['precio_base']) / inferencia_df['precio_base']) * 100

# Variable precio_competencia y ratio_precio
competidores = ['Amazon', 'Decathlon', 'Deporvillage']
for col in competidores:
    if col in inferencia_df.columns:
        inferencia_df[col] = pd.to_numeric(inferencia_df[col], errors='coerce')
inferencia_df['precio_competencia'] = inferencia_df[competidores].mean(axis=1)
inferencia_df['ratio_precio'] = inferencia_df['precio_venta'] / inferencia_df['precio_competencia']
inferencia_df = inferencia_df.drop(columns=competidores)


In [6]:
# Copia de variables categóricas con sufijo _h
inferencia_df['nombre_h'] = inferencia_df['nombre']
inferencia_df['categoria_h'] = inferencia_df['categoria']
inferencia_df['subcategoria_h'] = inferencia_df['subcategoria']

# One hot encoding con las mismas columnas que el entrenamiento
df_cols = pd.read_csv('../data/processed/df.csv', nrows=1).columns
categorical_cols = [col for col in df_cols if col.startswith('nombre_h_') or col.startswith('categoria_h_') or col.startswith('subcategoria_h_')]
inferencia_df = pd.get_dummies(inferencia_df, columns=['nombre_h', 'categoria_h', 'subcategoria_h'], drop_first=False)
# Añadir columnas faltantes y reordenar
for col in categorical_cols:
    if col not in inferencia_df.columns:
        inferencia_df[col] = 0
inferencia_df = inferencia_df.reindex(columns=list(df_cols), fill_value=0)


In [7]:
# Eliminar registros de octubre y dejar solo noviembre
inferencia_df = inferencia_df[inferencia_df['mes'] == 11].copy()

# Guardar el dataframe transformado
inferencia_df.to_csv('../data/processed/inferencia_df_transformado.csv', index=False)
print('Dataframe de inferencia transformado y guardado en data/processed/inferencia_df_transformado.csv')

Dataframe de inferencia transformado y guardado en data/processed/inferencia_df_transformado.csv


In [8]:
inferencia_df

,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,...,subcategoria_h_Esterilla Yoga,subcategoria_h_Mancuernas Ajustables,subcategoria_h_Mochila Trekking,subcategoria_h_Pesa Rusa,subcategoria_h_Pesas Casa,subcategoria_h_Rodillera Yoga,subcategoria_h_Ropa Montaña,subcategoria_h_Ropa Running,subcategoria_h_Zapatillas Running,subcategoria_h_Zapatillas Trail
168,2025-11-01,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.00,NaN,...,False,False,False,False,False,False,False,False,True,False
169,2025-11-01,PROD_002,Adidas Ultraboost 23,Running,Zapatillas Running,135,True,NaN,135.00,NaN,...,False,False,False,False,False,False,False,False,True,False
170,2025-11-01,PROD_003,Asics Gel Nimbus 25,Running,Zapatillas Running,85,False,NaN,86.39,NaN,...,False,False,False,False,False,False,False,False,True,False
171,2025-11-01,PROD_004,New Balance Fresh Foam X 1080v12,Running,Zapatillas Running,75,False,NaN,74.09,NaN,...,False,False,False,False,False,False,False,False,True,False
172,2025-11-01,PROD_005,Nike Dri-FIT Miler,Running,Ropa Running,35,False,NaN,34.76,NaN,...,False,False,False,False,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
883,2025-11-30,PROD_020,Quechua MH500,Outdoor,Ropa Montaña,80,False,NaN,79.64,NaN,...,False,False,False,False,False,False,True,False,False,False
884,2025-11-30,PROD_021,Manduka PRO Yoga Mat,Wellness,Esterilla Yoga,130,True,NaN,130.00,NaN,...,True,False,False,False,False,False,False,False,False,False
885,2025-11-30,PROD_022,Gaiam Premium Yoga Block,Wellness,Bloque Yoga,20,False,NaN,20.18,NaN,...,False,False,False,False,False,False,False,False,False,False
886,2025-11-30,PROD_023,Liforme Yoga Pad,Wellness,Rodillera Yoga,35,False,NaN,34.79,NaN,...,False,False,False,False,False,True,False,False,False,False


In [12]:
inferencia_df.columns

Index(['fecha', 'producto_id', 'nombre', 'categoria', 'subcategoria',
       'precio_base', 'es_estrella', 'unidades_vendidas', 'precio_venta',
       'ingresos', 'dia_semana', 'año', 'mes', 'dia_mes', 'es_fin_de_semana',
       'es_festivo', 'es_black_friday', 'es_cyber_monday', 'semana_año',
       'dia_año', 'es_primer_dia_mes', 'es_ultimo_dia_mes', 'trimestre',
       'es_festivo_o_finde', 'unidades_vendidas_lag1',
       'unidades_vendidas_lag2', 'unidades_vendidas_lag3',
       'unidades_vendidas_lag4', 'unidades_vendidas_lag5',
       'unidades_vendidas_lag6', 'unidades_vendidas_lag7',
       'unidades_vendidas_mm7', 'descuento_porcentaje', 'precio_competencia',
       'ratio_precio', 'nombre_h_Adidas Own The Run Jacket',
       'nombre_h_Adidas Ultraboost 23', 'nombre_h_Asics Gel Nimbus 25',
       'nombre_h_Bowflex SelectTech 552', 'nombre_h_Columbia Silver Ridge',
       'nombre_h_Decathlon Bandas Elásticas Set', 'nombre_h_Domyos BM900',
       'nombre_h_Domyos Kit Mancuernas